# Phase 3 — Gold Layer Validation

Run this notebook **after** the Full Load or Incremental Load notebook finishes and the Gold refresh completes.

It validates row grain, uniqueness, dimension relationships, and aggregate table keys. It does not modify production tables.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

SILVER = "silver_weather"
DIM_CITY = "dim_city"
DIM_DATE = "dim_date"
FACT = "fact_weather"
DAILY = "gold_daily_weather"
MONTHLY = "gold_monthly_climate"
ANNUAL = "gold_annual_climate"
EXTREME = "gold_extreme_weather"

required_tables = [SILVER, DIM_CITY, DIM_DATE, FACT, DAILY, MONTHLY, ANNUAL, EXTREME]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    raise ValueError(f"Missing required tables: {missing}")

print("All required Silver and Gold tables exist.")


All required Silver and Gold tables exist.


In [0]:
silver = spark.table(SILVER)
fact = spark.table(FACT)
dim_city = spark.table(DIM_CITY)
dim_date = spark.table(DIM_DATE)
daily = spark.table(DAILY)
monthly = spark.table(MONTHLY)
annual = spark.table(ANNUAL)
extreme = spark.table(EXTREME)

silver_rows = silver.count()
fact_rows = fact.count()
fact_unique_keys = fact.select("city_key", "timestamp").distinct().count()
city_rows = dim_city.count()
city_unique_keys = dim_city.select("city_key").distinct().count()
date_rows = dim_date.count()
date_unique_keys = dim_date.select("date_key").distinct().count()

print("Silver rows:", silver_rows)
print("Fact rows:", fact_rows)
print("Fact duplicate logical keys:", fact_rows - fact_unique_keys)
print("City dimension rows / unique keys:", city_rows, "/", city_unique_keys)
print("Date dimension rows / unique keys:", date_rows, "/", date_unique_keys)

silver.select(
    F.countDistinct("city").alias("silver_cities"),
    F.min("timestamp").alias("first_timestamp"),
    F.max("timestamp").alias("last_timestamp")
).show(truncate=False)

fact.select(
    F.min("timestamp").alias("first_timestamp"),
    F.max("timestamp").alias("last_timestamp"),
    F.min("observation_year").alias("first_partition_year"),
    F.max("observation_year").alias("last_partition_year")
).show(truncate=False)


Silver rows: 3550176
Fact rows: 3550176
Fact duplicate logical keys: 0
City dimension rows / unique keys: 9 / 9
Date dimension rows / unique keys: 16436 / 16436
+-------------+-------------------+-------------------+
|silver_cities|first_timestamp    |last_timestamp     |
+-------------+-------------------+-------------------+
|9            |1981-01-01 00:00:00|2025-12-31 23:00:00|
+-------------+-------------------+-------------------+

+-------------------+-------------------+--------------------+-------------------+
|first_timestamp    |last_timestamp     |first_partition_year|last_partition_year|
+-------------------+-------------------+--------------------+-------------------+
|1981-01-01 00:00:00|2025-12-31 23:00:00|1981                |2025               |
+-------------------+-------------------+--------------------+-------------------+



In [0]:
# Referential integrity: every fact foreign key should match its dimension.
orphan_city_keys = (
    fact.select("city_key").distinct()
    .join(dim_city.select("city_key").distinct(), on="city_key", how="left_anti")
    .count()
)

orphan_date_keys = (
    fact.select("date_key").distinct()
    .join(dim_date.select("date_key").distinct(), on="date_key", how="left_anti")
    .count()
)

# Grain checks for each aggregate.
daily_rows = daily.count()
daily_unique = daily.select("city_key", "date_key").distinct().count()
monthly_rows = monthly.count()
monthly_unique = monthly.select("city_key", "date_key").distinct().count()
annual_rows = annual.count()
annual_unique = annual.select("city_key", "calendar_year").distinct().count()
extreme_rows = extreme.count()
extreme_unique = extreme.select("city_key", "date_key").distinct().count()

checks = {
    "Fact row count equals Silver row count": fact_rows == silver_rows,
    "No duplicate fact city_key + timestamp": fact_rows == fact_unique_keys,
    "dim_city city_key is unique": city_rows == city_unique_keys,
    "dim_date date_key is unique": date_rows == date_unique_keys,
    "No orphan city foreign keys": orphan_city_keys == 0,
    "No orphan date foreign keys": orphan_date_keys == 0,
    "Daily grain unique": daily_rows == daily_unique,
    "Monthly grain unique": monthly_rows == monthly_unique,
    "Annual grain unique": annual_rows == annual_unique,
    "Extreme-weather daily grain unique": extreme_rows == extreme_unique,
}

print("Orphan city keys:", orphan_city_keys)
print("Orphan date keys:", orphan_date_keys)
print("Daily rows:", daily_rows, "| duplicate keys:", daily_rows - daily_unique)
print("Monthly rows:", monthly_rows, "| duplicate keys:", monthly_rows - monthly_unique)
print("Annual rows:", annual_rows, "| duplicate keys:", annual_rows - annual_unique)
print("Extreme rows:", extreme_rows, "| duplicate keys:", extreme_rows - extreme_unique)

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL") + " — " + name)

assert all(checks.values()), "One or more Gold validation checks failed."
print("GOLD VALIDATION PASSED")


Orphan city keys: 0
Orphan date keys: 0
Daily rows: 147924 | duplicate keys: 0
Monthly rows: 4860 | duplicate keys: 0
Annual rows: 405 | duplicate keys: 0
Extreme rows: 147924 | duplicate keys: 0
PASS — Fact row count equals Silver row count
PASS — No duplicate fact city_key + timestamp
PASS — dim_city city_key is unique
PASS — dim_date date_key is unique
PASS — No orphan city foreign keys
PASS — No orphan date foreign keys
PASS — Daily grain unique
PASS — Monthly grain unique
PASS — Annual grain unique
PASS — Extreme-weather daily grain unique
GOLD VALIDATION PASSED


In [0]:
print("===== FACT SCHEMA =====")
fact.printSchema()

print("===== CITY DIMENSION =====")
display(dim_city.orderBy("city"))

print("===== DATE DIMENSION SAMPLE =====")
display(dim_date.orderBy("date").limit(15))

print("===== DAILY WEATHER SAMPLE =====")
display(daily.orderBy("date_key", "city_key").limit(20))

print("===== GOLD EXECUTION LOGS =====")
display(
    spark.table("pipeline_execution_logs")
    .filter(col("layer") == "Gold")
    .orderBy(col("start_time").desc())
    .limit(30)
)

print("===== GOLD ERRORS (if any) =====")
display(
    spark.table("pipeline_error_logs")
    .filter(col("layer") == "Gold")
    .orderBy(col("error_timestamp").desc())
    .limit(30)
)


===== FACT SCHEMA =====
root
 |-- city_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- observation_year: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- source: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)

===== CITY DIMENSION =====


city_key,city,country,region,latitude,longitude
7434826648374848820,Berlin,Germany,Central Europe,52.5,13.5
-1898283651665344956,Copenhagen,Denmark,Northern Europe,55.75,12.5
6569788255095896663,Helsinki,Finland,Northern Europe,60.25,25.0
8472026731766187079,London,United Kingdom,Western Europe,51.5,-0.25
6554272843072168102,Madrid,Spain,Southern Europe,40.5,-3.75
-3419634064044749622,Oslo,Norway,Northern Europe,60.0,10.75
-9203903327554242704,Rome,Italy,Southern Europe,42.0,12.5
3335387132003167273,Stockholm,Sweden,Northern Europe,59.25,18.0
-8189021107459632899,Warsaw,Poland,Eastern Europe,52.25,21.0


===== DATE DIMENSION SAMPLE =====


date_key,date,day,month,month_name,quarter,year,decade,season
19810101,1981-01-01,1,1,January,1,1981,1980,Winter
19810102,1981-01-02,2,1,January,1,1981,1980,Winter
19810103,1981-01-03,3,1,January,1,1981,1980,Winter
19810104,1981-01-04,4,1,January,1,1981,1980,Winter
19810105,1981-01-05,5,1,January,1,1981,1980,Winter
19810106,1981-01-06,6,1,January,1,1981,1980,Winter
19810107,1981-01-07,7,1,January,1,1981,1980,Winter
19810108,1981-01-08,8,1,January,1,1981,1980,Winter
19810109,1981-01-09,9,1,January,1,1981,1980,Winter
19810110,1981-01-10,10,1,January,1,1981,1980,Winter


===== DAILY WEATHER SAMPLE =====


city_key,date_key,avg_temperature_c,min_temperature_c,max_temperature_c,total_precipitation_mm,total_rain_mm,total_snowfall_cm,avg_relative_humidity_pct,max_wind_speed_kmh,max_hourly_precipitation_mm,heavy_rain_hours,strong_wind_hours,hourly_observations
-9203903327554242704,19810101,3.6874999999999987,-1.9,8.2,0.0,0.0,0.0,82.875,8.3,0.0,0,0,24
-8189021107459632899,19810101,2.375,0.9,4.9,6.699999999999999,5.3999999999999995,0.9100000000000001,80.41666666666667,36.4,1.2,0,0,24
-3419634064044749622,19810101,-2.15,-5.3,1.5,0.1,0.0,0.07,70.70833333333333,13.0,0.1,0,0,24
-1898283651665344956,19810101,2.6625,1.2,4.0,1.1,0.9,0.14,80.04166666666667,45.8,0.6,0,0,24
3335387132003167273,19810101,0.28750000000000003,-1.0,2.3,1.4999999999999998,0.1,1.12,78.125,32.2,0.3,0,0,24
6554272843072168102,19810101,1.6833333333333336,-5.3,10.6,0.0,0.0,0.0,69.58333333333333,9.4,0.0,0,0,24
6569788255095896663,19810101,-0.37500000000000006,-3.5,1.8,4.4,1.0,2.45,96.83333333333333,20.5,1.4,0,0,24
7434826648374848820,19810101,2.6666666666666665,1.4,5.2,5.2,5.1,0.07,81.70833333333333,38.5,1.4,0,0,24
8472026731766187079,19810101,5.970833333333332,4.7,7.2,0.0,0.0,0.0,70.33333333333333,28.3,0.0,0,0,24
-9203903327554242704,19810102,6.245833333333333,1.7,12.6,0.0,0.0,0.0,75.75,11.5,0.0,0,0,24


===== GOLD EXECUTION LOGS =====


run_id,batch_id,layer,load_type,parameter_or_file,start_time,end_time,status,rows_inserted,rows_updated,error_message
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_gold_extreme_weather_6df04201,Gold,FULL,gold_extreme_weather,2026-10-25T10:14:15.532Z,2026-10-25T10:14:19.760Z,SUCCESS,147924,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_gold_annual_climate_6df04201,Gold,FULL,gold_annual_climate,2026-10-25T10:14:09.050Z,2026-10-25T10:14:13.354Z,SUCCESS,405,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_gold_monthly_climate_6df04201,Gold,FULL,gold_monthly_climate,2026-10-25T10:14:00.971Z,2026-10-25T10:14:06.872Z,SUCCESS,4860,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_gold_daily_weather_6df04201,Gold,FULL,gold_daily_weather,2026-10-25T10:13:52.163Z,2026-10-25T10:13:58.050Z,SUCCESS,147924,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_fact_weather_6df04201,Gold,FULL,fact_weather,2026-10-25T10:13:34.299Z,2026-10-25T10:13:49.928Z,SUCCESS,3550176,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_dim_date_6df04201,Gold,FULL,dim_date,2026-10-25T10:13:21.813Z,2026-10-25T10:13:31.807Z,SUCCESS,16436,0,null
6df04201-d87b-4e2e-aae2-b3043720a9b0,gold_dim_city_6df04201,Gold,FULL,dim_city,2026-10-25T10:12:55.817Z,2026-10-25T10:13:17.331Z,SUCCESS,9,0,null


===== GOLD ERRORS (if any) =====


error_id,run_id,batch_id,layer,source_file,error_type,error_message,error_timestamp
